## Project: Scrape ICE Detention Facilities Website

**Description:** Scrape the ICE website for information about its detention facilities, including names and addresses.

**Goal:** Map the locations of ICE detention facilities to see how widely they are spread across the country. For example:

- Which state has the most ICE detention facilities?
- Can we map them all?
- What other patterns emerge from the map?



In [ ]:
import requests

In [ ]:
base_url = 'https://www.ice.gov/detention-facilities'

response = requests.get(base_url)

response.raise_for_status(), response.status_code


In [ ]:
# print(response.text[:5000])

In [ ]:
# !pip install --quiet beautifulsoup4
from bs4 import BeautifulSoup

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')


In [ ]:
# print(soup.prettify())

In [ ]:
# find the number of pages
page_list=[]
## find number of pages on this pages using this page source:
for li in soup.find_all('li', class_='usa-pagination__item usa-pagination__page-no'):
	pages=li.find('a')['href']
	page_list.append(pages)

print(page_list)

In [ ]:
pagination_bar=soup.find("nav", class_="usa-pagination")

In [ ]:
pagination_bar.find_all("li")[0]

In [ ]:
page_num=[]
for a in pagination_bar.find_all("a", class_="usa-pagination__button"):
    page_num.append(a["href"])


In [ ]:
## other method
last_link = pagination_bar.find("a", attrs={"aria-label": "Last page"})
print(last_link['href'])
last_page = int(last_link["href"].split("=")[1])
print(last_page)  # 7


In [ ]:
for page in range(last_page + 1):
    print(f"Fetching page {page}")

#### Scrape grids

In [ ]:
soup.prettify()

In [ ]:
grids= soup.find_all("li", class_="grid")

links=[]
for grid in grids:
    name_tag = grid.find('span', class_='field-content')
    address_tag = grid.find('p', class_='address')
    link_tag = "https://www.ice.gov/"+grid.find('a')['href']
    image="https://www.ice.gov/"+grid.find('img')['src']
    links.append({
        'name': name_tag.text if name_tag else None,
        'address': address_tag.text if address_tag else None,
        'link': link_tag if link_tag else None,
        'image': image
    })


In [ ]:
import pandas as pd


In [ ]:

df=pd.DataFrame(links)
df.head()



In [ ]:
import time

def scrape_page(page_num):
    url = f"{base_url}?page={page_num}"
    headers = {
	'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")

    rows = []
    for grid in soup.find_all("li", class_="grid"):
        name_tag = grid.find("span", class_="field-content")
        address_tag = grid.find("p", class_="address")
        link_tag = grid.find("a")
        img_tag = grid.find("img")

        rows.append({
            "name": name_tag.get_text(strip=True) if name_tag else None,
            "address": address_tag.get_text(" ", strip=True) if address_tag else None,
            "link": "https://www.ice.gov" + link_tag["href"] if link_tag else None,
            "image": "https://www.ice.gov" + img_tag["src"] if img_tag else None,
        })
    return pd.DataFrame(rows)


dfs = []
for page in range(last_page + 1):   # 0 through last_page
    dfs.append(scrape_page(page))
    time.sleep(1)                   # be polite

df = pd.concat(dfs, ignore_index=True)
df.shape

In [ ]:
df.head()

In [ ]:
headers = {
	'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
    }

import re


for index, row in df.iterrows():
    url = row["link"]
    name = row["name"]

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
    except Exception as e:
        # slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
        # new_url = "https://www.ice.gov/detain/detention-facilities/" + slug
        new_url = "https://www.ice.gov/detain/detention-facilities/" + name.replace(" ", "-").lower()
        df.loc[index, "link"] = new_url
        print(f"{name} {url}: {e}\n  -> replaced with {new_url}")

In [ ]:
headers = {
	'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
    }



# for index, row in df.head(2).iterrows():
#     url=row['link']
#     response=requests.get(url, headers=headers)
#     soup=BeautifulSoup(response.content, 'html.parser')

#     container = soup.find("div", class_="field--name-field-a11y-para-tabs-text-area")
#     text = container.get_text(" ", strip=True) if container else None    
    



def get_info(url):
    try:
        response = requests.get(url, headers=headers, timeout=30)
    except Exception:
        return None
    soup = BeautifulSoup(response.content, "html.parser")
    container = soup.find("div", class_="field--name-field-a11y-para-tabs-text-area")
    time.sleep(1)
    return container.get_text(" ", strip=True) if container else None

df["info"] = df["link"].apply(get_info)

In [ ]:
# First extract the phone number into a new column

# \( a literal ( (escaped, because bare ( starts a group)
# \d{3}	exactly 3 digits (the area code)
# \)	a literal )
# ?	an optional space (the ? makes the space appear 0 or 1 times)
# \d{3}	3 digits
# -	a literal hyphen
# \d{4}	4 digits

df["phone_numbers"] = df["info"].str.findall(r"\(\d{3}\) ?\d{3}-\d{4}")

In [ ]:
df['info'].isnull().sum()

In [ ]:
import re
import time
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}

def fetch(url):
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        return response
    except Exception as e:
        print(f"  failed {url}: {e}")
        return None

df["info"] = None

for index, row in df.iterrows():
    name = row["name"]
    response = fetch(row["link"])

    if response is None:
        new_url = "https://www.ice.gov/detain/detention-facilities/" + name.replace(" ", "-").lower()
        df.loc[index, "link"] = new_url
        print(f"{name} -> trying {new_url}")
        response = fetch(new_url)

    if response is not None:
        soup = BeautifulSoup(response.content, "html.parser")
        container = soup.find("div", class_="field--name-field-a11y-para-tabs-text-area")
        df.loc[index, "info"] = container.get_text(" ", strip=True) if container else None

    time.sleep(1)

In [ ]:
df['info'].isna().sum()

In [ ]:

import os


os.makedirs("images", exist_ok=True)

df.to_csv("scraped_data/detention_facilities.csv", index=False)

#### Geocoding

In [20]:
df.clean_address[0]

'20 Hobo Forks Road Natchez, MS 39120'

In [ ]:
df["zipcode"] = df["clean_address"].str.extract(r"(\d{5})(?:-\d{4})?\s*$") # zip plus 4-digit extension

In [23]:
df["state"] = df["clean_address"].str.extract(r"([A-Z]{2})\s+\d{5}")

In [ ]:
# re.sub(pattern, replacement, string)

def clean_address(addr):
    addr = re.sub(r"\s+United States\s*$", "", str(addr))  # drop trailing country
    addr = re.sub(r"\s+,", ",", addr)                      # "Natchez , MS" -> "Natchez, MS"
    return re.sub(r"\s+", " ", addr).strip()

In [ ]:
import re
import time
import requests
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import certifi
import ssl

ctx = ssl.create_default_context(cafile=certifi.where())


In [ ]:
## use census geocoder
def census_geocode(addr):
    r = requests.get(
        "https://geocoding.geo.census.gov/geocoder/locations/onelineaddress",
        params={"address": addr, "benchmark": "Public_AR_Current", "format": "json"},
        timeout=30,
    )
    matches = r.json()["result"]["addressMatches"]
    if matches:
        c = matches[0]["coordinates"]
        return c["y"], c["x"]
    return None, None


#  use nominatim
headers = {"User-Agent": "cuny-class-geocoder"}  # Nominatim requires one

def nominatim_geocode(addr):
    r = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": addr, "format": "json", "limit": 1, "countrycodes": "us"},
        headers=headers,
        timeout=30,
    )
    time.sleep(1)  # Nominatim's rule: max 1 request per second
    results = r.json()
    if results:
        return float(results[0]["lat"]), float(results[0]["lon"])
    return None, None


## the main geocoding function to call census and nominatim
def geocode(addr):
    lat, lon = census_geocode(addr)
    source = "census"
    if lat is None:
        lat, lon = nominatim_geocode(addr)
        source = "nominatim" if lat is not None else None
    return pd.Series({"lat": lat, "lon": lon, "geo_source": source})

In [ ]:
df["clean_address"] = df["address"].map(clean_address)

In [ ]:
df[["lat", "lon", "geo_source"]] = df["clean_address"].apply(geocode)

In [ ]:
df.geo_source.value_counts()

In [ ]:
df.lat.isna().sum()

In [ ]:
import plotly.express as px

mapped = df.dropna(subset=["lat", "lon"])  # skip rows that didn't geocode

fig = px.scatter_map(
    mapped,
    lat="lat",
    lon="lon",
    hover_name="name",
    hover_data={"clean_address": True, "lat": False, "lon": False},
    center={"lat": 39.8, "lon": -98.6},
    zoom=3,
    height=600,
)
fig.show()

In [ ]:
import plotly.express as px

mapped = df.dropna(subset=["lat", "lon"])  # skip rows that didn't geocode

fig = px.scatter_geo(
    mapped,
    lat="lat",
    lon="lon",
    hover_name="name",
    hover_data={"clean_address": True, "lat": False, "lon": False},
    labels={"clean_address": "Address"},
    scope="usa",
    height=600,
)
fig.show()
fig.write_html("facilities_map.html")

In [ ]:
fig.write_html("facilities_map.html")

#### Downloading images

In [ ]:
import pandas as pd
df=pd.read_csv("scraped_data/detention_facilities.csv")

df.shape

In [ ]:
os.makedirs("images", exist_ok=True)

In [ ]:
for index, row in df.head().iterrows():
    image_url = row["image"]
    if not image_url:
        continue

    filename = f"{index}_" +os.path.basename(image_url.split("/")[-1])
    filepath = os.path.join("images", filename)

    print(image_url, filename, filepath)

    try:
        response = requests.get(image_url, headers=headers, timeout=30)
        response.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(response.content)
        df.loc[index, "image_file"] = filepath

    except Exception as e:
        print(f"{row['name']} {image_url}: {e}")

    time.sleep(1)



In [ ]:
df.head()

#### Make a gif and a grid

In [ ]:
import math
from PIL import Image, ImageOps

thumbnail_size = (300, 200)   # width, height of each frame/tile

# Load every image that downloaded successfully
frames = []
for path in df["image_file"].dropna():
    try:
        img = Image.open(path).convert("RGB")
        frames.append(ImageOps.fit(img, thumbnail_size))   # crop/resize to exactly thumbnail_size
    except Exception as e:
        print(f"Skipping {path}: {e}")

print(f"{len(frames)} images loaded")

In [ ]:
frames[0].save( # uses the first image as a starting frame and write the file
    "facilities.gif",
    save_all=True, # save multiple frames
    append_images=frames[1:], # adds the remaining images
    duration=300,   # ms per frame
    loop=0,         # 0 = loop forever, loop=1 would loop once
    optimize=True
)

In [ ]:
thumbnail_size = (300, 200)   # width, height of each frame/tile

cols = 10
rows = math.ceil(len(frames) / cols)

grid = Image.new("RGB", (cols * thumbnail_size[0], rows * thumbnail_size[1]), "white") # creates a blnak white canvas big enough for the whole grid 

for i, frame in enumerate(frames):
    x = (i % cols) * thumbnail_size[0] ## give the x-coordinate of the top-left corner of the tile
    y = (i // cols) * thumbnail_size[1] ## give the y-coordinate of the top-left corner of the tile
    grid.paste(frame, (x, y)) # pastes the frame onto the grid at the calculated position

grid.save("facilities_grid.jpg", quality=85)